In [14]:
import cupy as cp
import numpy as np
from time import perf_counter
from pyqcu import define
from pyqcu import io
from pyqcu import qcu
import cupyx.scipy.sparse.linalg as csla
from typing import Callable, Tuple, List

print('My rank is ', define.rank)

My rank is  0


In [15]:
params = np.array([0]*define._PARAMS_SIZE_, dtype=np.int32)
params[define._LAT_X_] = 32
params[define._LAT_Y_] = 32
params[define._LAT_Z_] = 32
params[define._LAT_T_] = 32
params[define._LAT_XYZT_] = 1048576
params[define._GRID_X_] = 1
params[define._GRID_Y_] = 1
params[define._GRID_Z_] = 1
params[define._GRID_T_] = 1
params[define._PARITY_] = 0
params[define._NODE_RANK_] = 0
params[define._NODE_SIZE_] = 1
params[define._DAGGER_] = 0
params[define._MAX_ITER_] = 1e4
params[define._DATA_TYPE_] = 0
params[define._SET_INDEX_] = 2
params[define._SET_PLAN_] = 0
argv = np.array([0.0]*define._ARGV_SIZE_, dtype=np.float32)
argv[define._MASS_] = 0.0
argv[define._TOL_] = 1e-9
print("Parameters:", params)
print("Arguments:", argv)

Parameters: [     32      32      32      32 1048576       1       1       1       1
       0       0       1       0   10000       0       2       0]
Arguments: [0.e+00 1.e-09]


In [16]:
gauge_filename = f"quda_wilson-dslash-gauge_-{params[define._LAT_X_]}-{params[define._LAT_Y_]}-{params  [define._LAT_Z_]}-{params[define._LAT_T_]}-{params[define._LAT_XYZT_]}-{params[define._GRID_X_]}-{params[define._GRID_Y_]}-{params[define._GRID_Z_]}-{params[define._GRID_T_]}-{params[define._PARITY_]}-{params[define._NODE_RANK_]}-{params[define._NODE_SIZE_]}-{params[define._DAGGER_]}-f.bin"
print("Gauge filename:", gauge_filename)
gauge = cp.fromfile(gauge_filename, dtype=cp.complex64,
                    count=params[define._LAT_XYZT_]*define._LAT_DCC_)
gauge = io.gauge2ccdptzyx(gauge, params)
print("Gauge:", gauge)
print("Gauge data:", gauge.data)
print("Gauge shape:", gauge.shape)

Gauge filename: quda_wilson-dslash-gauge_-32-32-32-32-1048576-1-1-1-1-0-0-1-0-f.bin
U: [ 0.9888613 +0.02130369j  0.07348315-0.02076806j -0.09857995+0.07842391j
 -0.07511304-0.01086277j  0.9932049 -0.00666964j  0.04573338+0.07515373j
  0.10230845+0.07397576j -0.03536638+0.08011071j  0.9880291 -0.01380186j]
_U: [ 0.        +0.j          0.        +0.j          0.        +0.j
  0.        +0.j          0.        +0.j          0.        +0.j
  0.10230846+0.07397576j -0.03536638+0.08011071j  0.9880291 -0.01380186j]
Gauge: 37748736
Gauge: [[[[[[[[ 9.88861322e-01+2.13036854e-02j
         9.90962327e-01+5.69420774e-03j
         9.87317920e-01-1.84401460e-02j ...
         9.82378662e-01-1.55471310e-01j
         9.85994279e-01+8.39557722e-02j
         9.77860272e-01+1.58485785e-01j]
       [ 9.87088621e-01-4.39827666e-02j
         9.86678004e-01+7.97407404e-02j
         9.95163381e-01+4.71003056e-02j ...
         9.88952160e-01-4.90137562e-02j
         9.68354821e-01-9.38901827e-02j
         9.92

In [17]:
set_ptrs = np.array(params, dtype=np.int64)
print("Set pointers:", set_ptrs)
print("Set pointers data:", set_ptrs.data)
qcu.applyInitQcu(set_ptrs, params, argv)

Set pointers: [     32      32      32      32 1048576       1       1       1       1
       0       0       1       0   10000       0       2       0]
Set pointers data: <memory at 0x7f111eb3b040>
gridDim.x               :4096
blockDim.x              :128
host_params[_LAT_X_]    :16
host_params[_LAT_Y_]    :32
host_params[_LAT_Z_]    :32
host_params[_LAT_T_]    :32
host_params[_LAT_XYZT_] :524288
host_params[_GRID_X_]   :1
host_params[_GRID_Y_]   :1
host_params[_GRID_Z_]   :1
host_params[_GRID_T_]   :1
host_params[_PARITY_]   :0
host_params[_NODE_RANK_]:0
host_params[_NODE_SIZE_]:1
host_params[_DAGGER_]   :0
host_params[_MAX_ITER_] :10000
host_params[_SET_INDEX_]:2
host_params[_SET_PLAN_] :0
host_argv[_MASS_]       :0.000000e+00
host_argv[_TOL_]        :1.000000e-09
lat_2dim[_XY_]          :512
lat_2dim[_XZ_]          :512
lat_2dim[_XT_]          :512
lat_2dim[_YZ_]          :1024
lat_2dim[_YT_]          :1024
lat_2dim[_ZT_]          :1024
lat_3dim[_YZT_]         :32768
lat_3dim[_XZT

In [18]:
# give x, b, r, r_tilde, p, v, s, t
lat_t = params[define._LAT_T_]
lat_z = params[define._LAT_Z_]
lat_y = params[define._LAT_Y_]
lat_x = int(params[define._LAT_X_]/define._LAT_P_)
lat_d = define._LAT_D_
lat_s = define._LAT_S_
lat_p = define._LAT_P_
lat_c = define._LAT_C_
latt_shape = (lat_s, lat_c, lat_t, lat_z, lat_y, lat_x)
MAX_ITER = params[define._MAX_ITER_]
TOL = argv[define._TOL_]
N = params[define._LAT_XYZT_] * define._LAT_HALF_SC_
define._LAT_Ne_ = 24
define._LAT_Ne_ = 1
max_eigen_value = 1.578
degree=5

In [19]:
def matvec(src):
    print("norm src", cp.linalg.norm(src))
    dest = cp.zeros(N, cp.complex64)
    qcu.applyWilsonCgDslashQcu(dest, src, gauge, set_ptrs, params)
    print("norm dest", cp.linalg.norm(dest))
    return dest


def chebyshev_matvec(src, degree=degree, matvec=matvec):
    print("norm src", cp.linalg.norm(src))
    T = cp.zeros((N, degree+1), dtype=src.dtype)
    T[:, 0] = src
    if degree > 0:
        T[:, 1] = matvec(src)
    for k in range(2, degree+1):
        T[:, k] = 2 * matvec(T[:, k-1]) - T[:, k-2]
    print("norm dest", cp.linalg.norm(T[:, -1]))
    return T[:, -1]

In [20]:
# def power_iteration(matvec, v0, num_iter=params[define._MAX_ITER_], tol=argv[define._TOL_]):
#     v = v0
#     for i in range(num_iter):
#         Av = matvec(v)
#         v_new = Av / cp.linalg.norm(Av)
#         diff = cp.linalg.norm(v_new - v)
#         print(f"Iteration {i+1}: difference = {diff:.6e}")
#         if diff < tol:
#             print(
#                 f"Power Iteration converged after {i+1} iterations. Difference: {diff:.6e}")
#             break
#         v = v_new
#     eigenvalue = cp.dot(v.conj().T, matvec(v))
#     return eigenvalue, v
# def validate_power_iteration():
#     v0 = cp.random.randn(N).astype(cp.complex64)
#     eigenvalue, eigenvector = power_iteration(_matvec, v0)
#     print("Dominant eigenvalue:", eigenvalue)
#     print("Corresponding eigenvector:", eigenvector[:10])
#     Ax = _matvec(eigenvector)
#     eigenvalue_check = cp.dot(eigenvector.conj().T, Ax)
#     print(f"Eigenvalue verification (v^T A v): {eigenvalue_check}")
#     if cp.isclose(eigenvalue, eigenvalue_check):
#         print("Validation passed: Eigenvalue calculation is correct.")
#     else:
#         print("Validation failed: Eigenvalue calculation is incorrect.")
#     print("Eigenvector norm:", cp.linalg.norm(eigenvector))
#     return eigenvalue, eigenvector
# eigenvalue, eigenvector = validate_power_iteration()
# print("Diff:", cp.linalg.norm(_matvec(eigenvector) -
#       eigenvalue*eigenvector) / cp.linalg.norm(eigenvector))

In [21]:
# print("Eigenvalue:", eigenvalue)

In [22]:
# # bistabcg
# def dot(a, b):
#     cp.cuda.runtime.deviceSynchronize()
#     return cp.inner(a.flatten().conjugate(), b.flatten())


# def diff(a, b):
#     cp.cuda.runtime.deviceSynchronize()
#     return cp.linalg.norm(a - b) / cp.linalg.norm(b)


# def run(eigen_value):
#     t0 = perf_counter()
#     x = cp.random.rand(N).astype(cp.complex64)
#     print("shape:", x.shape)
#     b = cp.zeros(N, cp.complex64)  # must be zero
#     r = cp.zeros(N, cp.complex64)
#     r_tilde = cp.zeros(N, cp.complex64)
#     p = cp.zeros(N, cp.complex64)
#     s = cp.zeros(N, cp.complex64)
#     v = cp.zeros(N, cp.complex64)
#     t = cp.zeros(N, cp.complex64)
#     r = b - _matvec(x, eigen_value)
#     r_tilde = r
#     r_norm2 = 0
#     rho_prev = 1
#     rho = 0
#     alpha = 1
#     omega = 1
#     beta = 0
#     for i in range(MAX_ITER):
#         # print("## rho:", rho)
#         rho = dot(r_tilde, r)
#         # print("## beta:", beta)
#         beta = (rho / rho_prev) * (alpha / omega)
#         p = r + (p - v * omega) * beta
#         # v = A * p
#         v = _matvec(p, eigen_value)
#         # print("## alpha:", alpha)
#         alpha = rho / dot(r_tilde, v)
#         s = r - v * alpha
#         # t = A * s
#         t = _matvec(s, eigen_value)
#         # print("## omega:", omega)
#         omega = dot(t, s) / dot(t, t)
#         x = x + p * alpha + s * omega
#         r = s - t * omega
#         r_norm2 = cp.linalg.norm(r)
#         print("##{}# r_norm2:{}".format(i, r_norm2))
#         _diff = diff(_matvec(x, 0.0), eigen_value*x)
#         print("diff:", _diff)
#         # break
#         # if (r_norm2 < TOL or i == MAX_ITER - 1):
#         #     print("### turns:", i)
#         #     break
#         if (_diff < TOL or i == MAX_ITER - 1):
#             print("### turns:", i)
#             break
#         rho_prev = rho
#     # print(" diff:", diff(_matvec(x, eigen_value), b))
#     # print("x:", x)
#     t1 = perf_counter()
#     print("Time:", t1-t0)
#     return x, _diff


# eigen_value = eigenvalue
# eigen_vector, _diff = run(eigen_value)
# print("eigen_vector:", eigen_vector)
# print("eigen_vector shape:", eigen_vector.shape)
# print("diff:", _diff)

In [23]:
# print("Difference:", cp.linalg.norm(eigenvector-eigen_vector)/cp.linalg.norm(eigenvector))

In [24]:
# import cupyx.scipy.sparse.linalg as csla
# define._LAT_Ne_ = 24
# A = csla.LinearOperator((N,
#                         N), matvec=_matvec, dtype=cp.complex64)
# evals, evecs = csla.eigsh(a=A, k=define._LAT_Ne_, which="SA",
#                           return_eigenvectors=True)
# print("evals:", evals)
# print("evecs:", evecs)
# cp.linalg.norm(_matvec(evecs[:, 0])-evals[0] *
#                evecs[:, 0])/cp.linalg.norm(evals[0]*evecs[:, 0])

In [25]:
def shifted_power_iteration(
    matvec,
    n=N,
    num_eigenvalues=define._LAT_Ne_,
    max_iter=params[define._MAX_ITER_],
    tol=argv[define._TOL_]
):
    eigenvalues = []
    eigenvectors = []
    # Buffer for intermediate results
    v = cp.ndarray((n,), dtype=np.complex64)
    w = cp.ndarray((n,), dtype=np.complex64)
    # Store cumulative shift
    total_shift = 0.0

    for k in range(num_eigenvalues):
        # Initialize random vector
        v[:] = cp.random.rand(n) + 1j * cp.random.rand(n)
        # Orthogonalize against previously found eigenvectors
        for j in range(k):
            projection = cp.dot(cp.conj(eigenvectors[j]), v)
            v -= projection * eigenvectors[j]
        # Normalize
        v /= cp.linalg.norm(v)

        # Power iteration
        eigenvalue = None
        for iter_count in range(max_iter):
            # Matrix-vector multiplication
            w = matvec(v)
            # Apply shift
            w += total_shift * v
            # Calculate Rayleigh quotient
            new_eigenvalue = cp.dot(cp.conj(v), w)
            # Normalize
            norm = cp.linalg.norm(w)
            if norm < tol:  # Avoid zero vector
                v[:] = cp.random.rand(n) + 1j * cp.random.rand(n)
                continue
            w /= norm

            # Check convergence
            if eigenvalue is not None:
                diff = abs(new_eigenvalue - eigenvalue)
                print(f"Iteration {iter_count} for eigenvalue {k}: {diff:.8f}")
                if diff < tol:
                    break
            eigenvalue = new_eigenvalue
            v[:] = w

        # Store results and subtract shift
        if eigenvalue is not None:
            eigenvalue -= total_shift
            eigenvalues.append(eigenvalue)
            eigenvectors.append(v.copy())
            # Update shift for next eigenvalue
            total_shift = eigenvalue.real

    return eigenvalues, eigenvectors


# Calculate eigenvalues and eigenvectors
eigenvalues, eigenvectors = shifted_power_iteration(chebyshev_matvec)

# Verify results
print("Computed eigenvalues:")
for i, ev in enumerate(eigenvalues):
    print(f"λ_{i} = {ev:.8f}")
    # Verify eigenvector
    v = eigenvectors[i]
    w = cp.zeros_like(v)
    w = chebyshev_matvec(v)
    error = cp.linalg.norm(w - ev * v) / cp.linalg.norm(w)
    print(f"Relative error: {error:.2e}")

norm src 1.0000001
norm src 1.0000001
norm dest 0.5633551
norm src 0.5633551
norm dest 0.48805937
norm src 1.3958521
norm dest 0.8728315
norm src 1.8343229
norm dest 1.228031
norm src 3.2243724
norm dest 1.9371667
norm dest 5.1697655
norm src 1.0
norm src 1.0
norm dest 1.2270266
norm src 1.2270266
norm dest 0.70237064
norm src 1.8495578
norm dest 1.1190844
norm src 2.8212447
norm dest 1.6824939
norm src 4.622968
norm dest 2.6712818
norm dest 7.4846387
Iteration 1 for eigenvalue 0: 2.55389404
norm src 1.0
norm src 1.0
norm dest 1.2094581
norm src 1.2094581
norm dest 0.6680031
norm src 1.7962424
norm dest 1.0574508
norm src 2.8633478
norm dest 1.6219623
norm src 4.5235467
norm dest 2.5678456
norm dest 7.493277
Iteration 2 for eigenvalue 0: 1.46791232
norm src 1.0
norm src 1.0
norm dest 1.2090006
norm src 1.2090006
norm dest 0.65001
norm src 1.758831
norm dest 1.0093652
norm src 2.8829515
norm dest 1.5641123
norm src 4.407534
norm dest 2.4273252
norm dest 7.3372984
Iteration 3 for eigenva

In [26]:
eigenvalues

[array(6.4260025-1.662915j, dtype=complex64)]

In [47]:
eigenvectors[0]*1e6

array([122.83551  -125.4096j   ,  -7.9705005  +6.8944445j,
       -37.572735  -18.302063j , ...,  25.840395  +44.738625j ,
        52.09843   -72.03906j  ,  45.276997 -146.85518j  ],
      dtype=complex64)

In [46]:
matvec(eigenvectors[0]*1e6)

norm src 1000000.06
norm dest 1218092.5


array([154.75403 -166.35913j , -17.865658  +8.512257j,
       -69.29008  -50.705986j, ...,  25.176186 +76.56621j ,
        96.62988  -79.32627j , 118.92091 -266.27493j ], dtype=complex64)

In [45]:
matvec(eigenvectors[0])/1.2180924

norm src 1.0
norm dest 1.2180924


array([ 1.2704621e-04-1.3657351e-04j, -1.4666919e-05+6.9882003e-06j,
       -5.6884088e-05-4.1627369e-05j, ...,  2.0668529e-05+6.2857463e-05j,
        7.9328864e-05-6.5123371e-05j,  9.7628799e-05-2.1859992e-04j],
      dtype=complex64)

In [32]:
cp.linalg.norm(matvec(eigenvectors[0]))

norm src 1.0
norm dest 1.2180924


array(1.2180924, dtype=float32)

In [33]:
cp.linalg.norm(eigenvectors[0])

array(1., dtype=float32)

In [ ]:
eigenvectors[0]

norm src 1.0
norm dest 1.2180924


array([ 1.4962500e-04-1.5276049e-04j, -9.7088068e-06+8.3980713e-06j,
       -4.5767065e-05-2.2293605e-05j, ...,  3.1475989e-05+5.4495777e-05j,
        6.3460706e-05-8.7750232e-05j,  5.5151566e-05-1.7888319e-04j],
      dtype=complex64)

In [74]:
# import numpy as np

# def min_eigenvalue_power_iteration(A, num_iterations=100, tolerance=1e-10):
#     """
#     使用幂迭代法计算矩阵的最小本征值

#     参数:
#         A: numpy数组，输入矩阵（方阵）
#         num_iterations: 最大迭代次数
#         tolerance: 收敛容差

#     返回:
#         min_eigenvalue: 最小本征值的估计值
#         min_eigenvector: 对应的本征向量
#     """
#     # 计算矩阵的最大绝对本征值（用于位移变换）
#     n = A.shape[0]
#     v = np.random.rand(n)
#     max_eigenvalue = power_iteration(A, num_iterations, tolerance)[0]

#     # 进行位移变换：B = max_eigenvalue * I - A
#     # 这样B的最大本征值对应A的最小本征值
#     I = np.eye(n)
#     B = max_eigenvalue * I - A

#     # 对变换后的矩阵使用标准幂迭移法
#     v = np.random.rand(n)
#     v = v / np.linalg.norm(v)

#     for i in range(num_iterations):
#         v_old = v.copy()

#         # 幂迭代步骤
#         v = B @ v
#         v = v / np.linalg.norm(v)

#         # 检查收敛性
#         if np.allclose(v, v_old, rtol=tolerance):
#             break

#     # 计算最小本征值
#     min_eigenvalue = max_eigenvalue - (v.T @ B @ v) / (v.T @ v)

#     return min_eigenvalue, v

# def power_iteration(A, num_iterations=100, tolerance=1e-10):
#     """
#     标准幂迭代法计算最大本征值
#     """
#     n = A.shape[0]
#     v = np.random.rand(n)
#     v = v / np.linalg.norm(v)

#     for i in range(num_iterations):
#         v_old = v.copy()
#         v = A @ v
#         v = v / np.linalg.norm(v)

#         if np.allclose(v, v_old, rtol=tolerance):
#             break

#     eigenvalue = (v.T @ A @ v) / (v.T @ v)
#     return eigenvalue, v

# # 测试代码
# if __name__ == "__main__":
#     # 创建一个对称矩阵进行测试
#     A = np.array([[4, -1, 0],
#                   [-1, 4, -1],
#                   [0, -1, 4]])

#     min_eval, min_evec = min_eigenvalue_power_iteration(A)
#     print(f"最小本征值: {min_eval:.6f}")
#     print(f"对应的本征向量: {min_evec}")

#     # 验证结果
#     true_eigenvalues = np.linalg.eigvals(A)
#     print(f"numpy计算的所有本征值: {true_eigenvalues}")
#     print(f"实际最小本征值: {min(true_eigenvalues.real):.6f}")

In [75]:
# qcu.applyEndQcu(set_ptrs, params)